# Leakage-Safe Stock Forecasting with an LSTM

This notebook is a compact interface to the tested `stock_lstm` package. It demonstrates the complete workflow without duplicating production code or relying on hidden notebook state. The target is the next-session log return, and final results are compared with a persistence baseline.

## 1. Setup

Run `python -m pip install -e ".[dev]"` from the repository root before opening the notebook.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from stock_lstm.config import ExperimentConfig
from stock_lstm.data import download_market_data
from stock_lstm.features import build_features
from stock_lstm.pipeline import forecast_next_session, run_training

## 2. Load and inspect adjusted market data

The downloader validates OHLCV columns, sorts dates, removes duplicates, and caches the result in `data/raw/`. Set `refresh=True` only when you want a fresh download.

In [ ]:
config = ExperimentConfig(ticker="GOOGL", start="2015-01-01", epochs=40)
market_data = download_market_data(config.ticker, config.start, config.end)
market_data.tail()

In [ ]:
ax = market_data["close"].plot(figsize=(12, 5), title=f"{config.ticker} adjusted close")
ax.set_ylabel("Price (USD)")
plt.show()

## 3. Inspect causal features

Rolling indicators use only the current and earlier observations. During sequence creation, the row being predicted is excluded from the model input.

In [ ]:
features = build_features(market_data)
features.tail()

## 4. Train once and evaluate on the untouched test period

The pipeline uses chronological 70/15/15 splits. Scalers are fitted on training observations only, validation controls early stopping, and the final report compares the LSTM with `next close = latest close`.

In [ ]:
run_dir = Path("artifacts/notebook_googl")
result = run_training(market_data, config, run_dir)
result.metrics

In [ ]:
predictions_plot = plt.imread(run_dir / "test_predictions.png")
plt.figure(figsize=(14, 7))
plt.imshow(predictions_plot)
plt.axis("off")
plt.show()

## 5. Forecast the next observation

A forecast is an experiment output, not a trading recommendation. The date below uses the next business day and may need adjustment for exchange holidays.

In [ ]:
forecast = forecast_next_session(market_data, run_dir)
print(json.dumps(forecast, indent=2))